In [1]:
# ============================================================
# PERSPECTIVE-CONDITIONED SOFT-LABEL FEASIBILITY AUDIT
# ============================================================

import os
import numpy as np
import pandas as pd

In [2]:
# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_PATH = "../data/processed/mhs_main_experiment_annotations_with_split.csv"

ann = pd.read_csv(DATA_PATH)

print("Dataset shape:", ann.shape)
print("Unique comments:", ann["comment_id"].nunique())
print("Unique annotators:", ann["annotator_id"].nunique())

ann.head()

Dataset shape: (49433, 53)
Unique comments: 19761
Unique annotators: 7851


,comment_id,annotator_id,sentiment,respect,insult,humiliate,status,dehumanize,violence,genocide,...,target_origin_undocumented,target_origin_other,text_original,text_clean,is_women_targeted,is_immigrant_targeted,annotator_gender_group,annotator_ideology_group,target_type,split
0,47101,3379,4.0,4.0,4.0,4.0,4.0,4.0,0.0,0.0,...,0,0,Question: These 4 broads who criticize America...,Question: These 4 broads who criticize America...,0,1,men,slightly_conservative,immigrant_only,validation
1,43625,7365,2.0,3.0,2.0,1.0,2.0,0.0,0.0,0.0,...,1,0,It is about time for all illegals to go back t...,It is about time for all illegals to go back t...,0,1,men,neutral,immigrant_only,train
2,12538,488,4.0,4.0,4.0,4.0,4.0,4.0,4.0,1.0,...,0,0,For starters bend over the one in pink and kic...,For starters bend over the one in pink and kic...,1,0,women,neutral,women_only,test
3,40171,10363,4.0,4.0,4.0,3.0,3.0,3.0,0.0,1.0,...,0,0,"OMG! The EGO's of these young, young, inexperi...","OMG! The EGO's of these young, young, inexperi...",1,0,women,slightly_conservative,women_only,train
4,1006,10615,4.0,4.0,4.0,4.0,4.0,4.0,4.0,0.0,...,0,0,"Worthless whore, these tits with look nice wit...","Worthless whore, these tits with look nice wit...",1,0,women,extremely_liberal,women_only,train


In [3]:
# ============================================================
# 2. CHECK REQUIRED COLUMNS
# ============================================================

required_cols = [
    "comment_id",
    "annotator_id",
    "text_clean",
    "split",
    "hatespeech",
    "annotator_gender_group",
    "annotator_ideology_group"
]

missing_cols = [col for col in required_cols if col not in ann.columns]

print("Missing columns:", missing_cols)

if len(missing_cols) == 0:
    print("All required columns are present.")

Missing columns: []
All required columns are present.


In [4]:
# ============================================================
# 3. BASIC DEMOGRAPHIC ANNOTATION COUNTS
# ============================================================

print("Gender annotation counts:")
gender_annotation_counts = (
    ann["annotator_gender_group"]
    .value_counts(dropna=False)
    .reset_index()
)

gender_annotation_counts.columns = ["gender_group", "annotation_rows"]
display(gender_annotation_counts)


print("Ideology annotation counts:")
ideology_annotation_counts = (
    ann["annotator_ideology_group"]
    .value_counts(dropna=False)
    .reset_index()
)

ideology_annotation_counts.columns = ["ideology_group", "annotation_rows"]
display(ideology_annotation_counts)

Gender annotation counts:


,gender_group,annotation_rows
0,women,27716
1,men,21112
2,non_binary,362
3,prefer_not_to_say,189
4,self_describe,54


Ideology annotation counts:


,ideology_group,annotation_rows
0,liberal,12437
1,neutral,8222
2,slightly_liberal,7844
3,extremely_liberal,6667
4,conservative,5772
5,slightly_conservative,5418
6,extremely_conservative,1655
7,no_opinion,1401
8,unknown,17


In [5]:
# ============================================================
# 4. HELPER FUNCTIONS
# ============================================================

LABELS = [0, 1, 2]

def entropy_np(probs):
    probs = np.array(probs, dtype=float)
    probs = probs[probs > 0]

    if len(probs) == 0:
        return 0.0

    return -np.sum(probs * np.log2(probs))


def make_dist_from_group(g):
    counts = g["hatespeech"].value_counts(normalize=True)

    return pd.Series({
        "hatespeech_0_prob": counts.get(0, 0.0),
        "hatespeech_1_prob": counts.get(1, 0.0),
        "hatespeech_2_prob": counts.get(2, 0.0),
        "annotation_count": len(g),
        "split": g["split"].iloc[0],
        "text_clean": g["text_clean"].iloc[0],
    })


def build_perspective_softlabel_df(df, group_col, min_annotations=2):
    out = (
        df.groupby(["comment_id", group_col])
        .apply(make_dist_from_group)
        .reset_index()
    )

    out = out[out["annotation_count"] >= min_annotations].copy()

    out = out.rename(columns={group_col: "perspective"})

    out["perspective_type"] = group_col

    out["input_text"] = (
        "[" +
        out["perspective"].astype(str).str.upper() +
        "] " +
        out["text_clean"].astype(str)
    )

    out["entropy"] = out[
        [
            "hatespeech_0_prob",
            "hatespeech_1_prob",
            "hatespeech_2_prob"
        ]
    ].apply(
        lambda row: entropy_np(row.values),
        axis=1
    )

    out["distribution_pattern"] = out[
        [
            "hatespeech_0_prob",
            "hatespeech_1_prob",
            "hatespeech_2_prob"
        ]
    ].apply(
        lambda row: tuple(np.round(row.values.astype(float), 3)),
        axis=1
    )

    return out

In [6]:
# ============================================================
# 5. BUILD PERSPECTIVE-CONDITIONED SOFT-LABEL DATASETS
# ============================================================

gender_perspective_2 = build_perspective_softlabel_df(
    ann,
    "annotator_gender_group",
    min_annotations=2
)

gender_perspective_3 = build_perspective_softlabel_df(
    ann,
    "annotator_gender_group",
    min_annotations=3
)

ideology_perspective_2 = build_perspective_softlabel_df(
    ann,
    "annotator_ideology_group",
    min_annotations=2
)

ideology_perspective_3 = build_perspective_softlabel_df(
    ann,
    "annotator_ideology_group",
    min_annotations=3
)

print("Gender min 2:", gender_perspective_2.shape)
print("Gender min 3:", gender_perspective_3.shape)
print("Ideology min 2:", ideology_perspective_2.shape)
print("Ideology min 3:", ideology_perspective_3.shape)

/tmp/ipykernel_52638/112317447.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_dist_from_group)
/tmp/ipykernel_52638/112317447.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_dist_from_group)
/tmp/ipykernel_52638/112317447.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the gr

Gender min 2: (8089, 12)
Gender min 3: (1689, 12)
Ideology min 2: (3668, 12)
Ideology min 3: (556, 12)


/tmp/ipykernel_52638/112317447.py:33: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(make_dist_from_group)


In [7]:
# ============================================================
# 6. CAPACITY SUMMARY
# ============================================================

perspective_capacity = pd.DataFrame([
    {
        "perspective_type": "gender",
        "min_annotations": 2,
        "training_examples": len(gender_perspective_2),
        "unique_comments": gender_perspective_2["comment_id"].nunique(),
        "unique_perspectives": gender_perspective_2["perspective"].nunique()
    },
    {
        "perspective_type": "gender",
        "min_annotations": 3,
        "training_examples": len(gender_perspective_3),
        "unique_comments": gender_perspective_3["comment_id"].nunique(),
        "unique_perspectives": gender_perspective_3["perspective"].nunique()
    },
    {
        "perspective_type": "ideology",
        "min_annotations": 2,
        "training_examples": len(ideology_perspective_2),
        "unique_comments": ideology_perspective_2["comment_id"].nunique(),
        "unique_perspectives": ideology_perspective_2["perspective"].nunique()
    },
    {
        "perspective_type": "ideology",
        "min_annotations": 3,
        "training_examples": len(ideology_perspective_3),
        "unique_comments": ideology_perspective_3["comment_id"].nunique(),
        "unique_perspectives": ideology_perspective_3["perspective"].nunique()
    }
])

perspective_capacity

,perspective_type,min_annotations,training_examples,unique_comments,unique_perspectives
0,gender,2,8089,7582,4
1,gender,3,1689,1612,4
2,ideology,2,3668,3252,8
3,ideology,3,556,242,8


In [8]:
# ============================================================
# 7. EXAMPLES PER PERSPECTIVE
# ============================================================

gender_examples_by_perspective_2 = (
    gender_perspective_2
    .groupby("perspective")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotations=("annotation_count", "mean"),
        median_annotations=("annotation_count", "median"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values("examples", ascending=False)
)

ideology_examples_by_perspective_2 = (
    ideology_perspective_2
    .groupby("perspective")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotations=("annotation_count", "mean"),
        median_annotations=("annotation_count", "median"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values("examples", ascending=False)
)

print("Gender min 2 examples by perspective:")
display(gender_examples_by_perspective_2)

print("Ideology min 2 examples by perspective:")
display(ideology_examples_by_perspective_2)

Gender min 2 examples by perspective:


,perspective,examples,unique_comments,mean_annotations,median_annotations,mean_entropy,zero_entropy_percent
3,women,4850,4850,3.785361,2.0,0.381104,61.57
0,men,3200,3200,3.879688,2.0,0.362966,63.22
1,non_binary,28,28,3.607143,3.0,0.481488,46.43
2,prefer_not_to_say,11,11,3.181818,2.0,0.306450,72.73


Ideology min 2 examples by perspective:


,perspective,examples,unique_comments,mean_annotations,median_annotations,mean_entropy,zero_entropy_percent
3,liberal,1263,1263,4.656374,2.0,0.397215,59.38
4,neutral,655,655,5.267176,2.0,0.375651,61.22
7,slightly_liberal,604,604,5.360927,2.0,0.335077,65.07
2,extremely_liberal,418,418,6.129187,2.0,0.345541,64.59
0,conservative,336,336,6.276786,2.0,0.391234,59.23
6,slightly_conservative,261,261,7.245211,2.0,0.382189,59.00
1,extremely_conservative,74,74,6.783784,3.0,0.382700,55.41
5,no_opinion,57,57,7.087719,4.0,0.407062,50.88


In [9]:
# ============================================================
# 8. SAME SUMMARY FOR MIN 3
# ============================================================

gender_examples_by_perspective_3 = (
    gender_perspective_3
    .groupby("perspective")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotations=("annotation_count", "mean"),
        median_annotations=("annotation_count", "median"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values("examples", ascending=False)
)

ideology_examples_by_perspective_3 = (
    ideology_perspective_3
    .groupby("perspective")
    .agg(
        examples=("comment_id", "count"),
        unique_comments=("comment_id", "nunique"),
        mean_annotations=("annotation_count", "mean"),
        median_annotations=("annotation_count", "median"),
        mean_entropy=("entropy", "mean"),
        zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2))
    )
    .reset_index()
    .sort_values("examples", ascending=False)
)

print("Gender min 3 examples by perspective:")
display(gender_examples_by_perspective_3)

print("Ideology min 3 examples by perspective:")
display(ideology_examples_by_perspective_3)

Gender min 3 examples by perspective:


,perspective,examples,unique_comments,mean_annotations,median_annotations,mean_entropy,zero_entropy_percent
3,women,1073,1073,10.069897,3.0,0.518502,46.69
0,men,594,594,12.126263,3.0,0.470523,50.34
1,non_binary,17,17,4.647059,4.0,0.557745,35.29
2,prefer_not_to_say,5,5,4.600000,5.0,0.474190,60.00


Ideology min 3 examples by perspective:


,perspective,examples,unique_comments,mean_annotations,median_annotations,mean_entropy,zero_entropy_percent
3,liberal,144,144,25.298611,3.0,0.511680,40.97
7,slightly_liberal,83,83,26.457831,7.0,0.558870,33.73
4,neutral,75,75,30.533333,10.0,0.560688,33.33
2,extremely_liberal,63,63,29.396825,14.0,0.435495,50.79
0,conservative,57,57,27.210526,26.0,0.569379,33.33
6,slightly_conservative,57,57,26.017544,23.0,0.644759,22.81
1,extremely_conservative,41,41,10.634146,9.0,0.471214,41.46
5,no_opinion,36,36,10.055556,9.0,0.477848,38.89


In [10]:
# ============================================================
# 9. QUALITY SUMMARY FOR EACH PERSPECTIVE DATASET
# ============================================================

def perspective_quality_summary(df, dataset_name):
    return {
        "dataset": dataset_name,
        "examples": len(df),
        "unique_comments": df["comment_id"].nunique(),
        "unique_perspectives": df["perspective"].nunique(),
        "mean_annotations_per_example": df["annotation_count"].mean(),
        "median_annotations_per_example": df["annotation_count"].median(),
        "mean_entropy": df["entropy"].mean(),
        "median_entropy": df["entropy"].median(),
        "zero_entropy_percent": round((df["entropy"] == 0).mean() * 100, 2),
        "nonzero_entropy_percent": round((df["entropy"] > 0).mean() * 100, 2),
        "unique_distribution_patterns": df["distribution_pattern"].nunique(),
        "top_pattern_percent": round(
            df["distribution_pattern"].value_counts().iloc[0] / len(df) * 100,
            2
        ) if len(df) > 0 else np.nan
    }


perspective_quality = pd.DataFrame([
    perspective_quality_summary(gender_perspective_2, "gender_min2"),
    perspective_quality_summary(gender_perspective_3, "gender_min3"),
    perspective_quality_summary(ideology_perspective_2, "ideology_min2"),
    perspective_quality_summary(ideology_perspective_3, "ideology_min3"),
])

perspective_quality

,dataset,examples,unique_comments,unique_perspectives,mean_annotations_per_example,median_annotations_per_example,mean_entropy,median_entropy,zero_entropy_percent,nonzero_entropy_percent,unique_distribution_patterns,top_pattern_percent
0,gender_min2,8089,7582,4,3.821239,2.0,0.374174,0.000000,62.18,37.82,119,50.36
1,gender_min3,1689,1612,4,10.722321,3.0,0.501892,0.426229,47.90,52.10,119,40.79
2,ideology_min2,3668,3252,8,5.462650,2.0,0.375486,0.000000,60.99,39.01,237,48.58
3,ideology_min3,556,242,8,24.843525,8.0,0.531087,0.538545,37.23,62.77,234,25.00


In [11]:
# ============================================================
# 10. SPLIT SUMMARY
# ============================================================

def split_summary(df, dataset_name):
    return (
        df.groupby("split")
        .agg(
            examples=("comment_id", "count"),
            unique_comments=("comment_id", "nunique"),
            mean_annotation_count=("annotation_count", "mean"),
            mean_entropy=("entropy", "mean"),
            zero_entropy_percent=("entropy", lambda x: round((x == 0).mean() * 100, 2)),
            unique_perspectives=("perspective", "nunique")
        )
        .reset_index()
        .assign(dataset=dataset_name)
    )


perspective_split_summary = pd.concat([
    split_summary(gender_perspective_2, "gender_min2"),
    split_summary(gender_perspective_3, "gender_min3"),
    split_summary(ideology_perspective_2, "ideology_min2"),
    split_summary(ideology_perspective_3, "ideology_min3"),
], ignore_index=True)

perspective_split_summary

,split,examples,unique_comments,mean_annotation_count,mean_entropy,zero_entropy_percent,unique_perspectives,dataset
0,test,1224,1140,4.035131,0.376000,62.09,4,gender_min2
1,train,5672,5316,3.783850,0.374394,62.17,4,gender_min2
2,validation,1193,1126,3.779547,0.371255,62.36,3,gender_min2
3,test,260,249,11.580769,0.516244,46.92,4,gender_min3
4,train,1168,1115,10.662671,0.502197,47.86,4,gender_min3
5,validation,261,248,10.134100,0.486233,49.04,3,gender_min3
6,test,547,490,6.012797,0.353236,62.71,8,ideology_min2
7,train,2552,2261,5.410266,0.378716,60.78,8,ideology_min2
8,validation,569,501,5.168717,0.382391,60.28,8,ideology_min2
9,test,80,37,29.437500,0.540252,32.50,8,ideology_min3


In [12]:
# ============================================================
# 11. DISTRIBUTION PATTERN SUMMARY
# ============================================================

def top_patterns(df, dataset_name, top_n=10):
    out = (
        df["distribution_pattern"]
        .value_counts()
        .head(top_n)
        .reset_index()
    )

    out.columns = ["distribution_pattern", "examples"]
    out["percent"] = round(out["examples"] / len(df) * 100, 2)
    out["dataset"] = dataset_name

    return out[
        [
            "dataset",
            "distribution_pattern",
            "examples",
            "percent"
        ]
    ]


pattern_summary = pd.concat([
    top_patterns(gender_perspective_2, "gender_min2"),
    top_patterns(gender_perspective_3, "gender_min3"),
    top_patterns(ideology_perspective_2, "ideology_min2"),
    top_patterns(ideology_perspective_3, "ideology_min3"),
], ignore_index=True)

pattern_summary

,dataset,distribution_pattern,examples,percent
0,gender_min2,"(1.0, 0.0, 0.0)",4074,50.36
1,gender_min2,"(0.5, 0.0, 0.5)",1325,16.38
2,gender_min2,"(0.0, 0.0, 1.0)",904,11.18
3,gender_min2,"(0.5, 0.5, 0.0)",551,6.81
4,gender_min2,"(0.0, 0.5, 0.5)",322,3.98
5,gender_min2,"(0.667, 0.0, 0.333)",222,2.74
6,gender_min2,"(0.333, 0.0, 0.667)",190,2.35
7,gender_min2,"(0.667, 0.333, 0.0)",123,1.52
8,gender_min2,"(0.333, 0.333, 0.333)",75,0.93
9,gender_min2,"(0.0, 1.0, 0.0)",52,0.64


In [13]:
# ============================================================
# 12. SAVE PERSPECTIVE DATASETS
# ============================================================

os.makedirs("../data/processed", exist_ok=True)

gender_perspective_2_path = "../data/processed/gender_perspective_hatespeech_softlabels_min2.csv"
gender_perspective_3_path = "../data/processed/gender_perspective_hatespeech_softlabels_min3.csv"
ideology_perspective_2_path = "../data/processed/ideology_perspective_hatespeech_softlabels_min2.csv"
ideology_perspective_3_path = "../data/processed/ideology_perspective_hatespeech_softlabels_min3.csv"

gender_perspective_2.to_csv(gender_perspective_2_path, index=False)
gender_perspective_3.to_csv(gender_perspective_3_path, index=False)
ideology_perspective_2.to_csv(ideology_perspective_2_path, index=False)
ideology_perspective_3.to_csv(ideology_perspective_3_path, index=False)

print("Saved:")
print(gender_perspective_2_path)
print(gender_perspective_3_path)
print(ideology_perspective_2_path)
print(ideology_perspective_3_path)

Saved:
../data/processed/gender_perspective_hatespeech_softlabels_min2.csv
../data/processed/gender_perspective_hatespeech_softlabels_min3.csv
../data/processed/ideology_perspective_hatespeech_softlabels_min2.csv
../data/processed/ideology_perspective_hatespeech_softlabels_min3.csv


In [14]:
# ============================================================
# 13. SAVE AUDIT REPORT
# ============================================================

os.makedirs("../results/tables", exist_ok=True)

report_path = "../results/tables/perspective_conditioned_softlabel_feasibility_audit.txt"

with open(report_path, "w", encoding="utf-8") as f:
    f.write("PERSPECTIVE-CONDITIONED SOFT-LABEL FEASIBILITY AUDIT\n")
    f.write("=" * 90 + "\n\n")

    f.write("1. DATASET SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(f"Annotation rows: {len(ann)}\n")
    f.write(f"Unique comments: {ann['comment_id'].nunique()}\n")
    f.write(f"Unique annotators: {ann['annotator_id'].nunique()}\n\n")

    f.write("2. ANNOTATION-LEVEL DEMOGRAPHIC COUNTS\n")
    f.write("-" * 90 + "\n")
    f.write("Gender counts:\n")
    f.write(gender_annotation_counts.to_string(index=False))
    f.write("\n\nIdeology counts:\n")
    f.write(ideology_annotation_counts.to_string(index=False))
    f.write("\n\n")

    f.write("3. PERSPECTIVE CAPACITY\n")
    f.write("-" * 90 + "\n")
    f.write(perspective_capacity.to_string(index=False))
    f.write("\n\n")

    f.write("4. GENDER MIN 2 EXAMPLES BY PERSPECTIVE\n")
    f.write("-" * 90 + "\n")
    f.write(gender_examples_by_perspective_2.to_string(index=False))
    f.write("\n\n")

    f.write("5. GENDER MIN 3 EXAMPLES BY PERSPECTIVE\n")
    f.write("-" * 90 + "\n")
    f.write(gender_examples_by_perspective_3.to_string(index=False))
    f.write("\n\n")

    f.write("6. IDEOLOGY MIN 2 EXAMPLES BY PERSPECTIVE\n")
    f.write("-" * 90 + "\n")
    f.write(ideology_examples_by_perspective_2.to_string(index=False))
    f.write("\n\n")

    f.write("7. IDEOLOGY MIN 3 EXAMPLES BY PERSPECTIVE\n")
    f.write("-" * 90 + "\n")
    f.write(ideology_examples_by_perspective_3.to_string(index=False))
    f.write("\n\n")

    f.write("8. PERSPECTIVE SOFT-LABEL QUALITY\n")
    f.write("-" * 90 + "\n")
    f.write(perspective_quality.to_string(index=False))
    f.write("\n\n")

    f.write("9. SPLIT SUMMARY\n")
    f.write("-" * 90 + "\n")
    f.write(perspective_split_summary.to_string(index=False))
    f.write("\n\n")

    f.write("10. TOP DISTRIBUTION PATTERNS\n")
    f.write("-" * 90 + "\n")
    f.write(pattern_summary.to_string(index=False))
    f.write("\n\n")


print("Saved:", report_path)
print(open(report_path, encoding="utf-8").read())

Saved: ../results/tables/perspective_conditioned_softlabel_feasibility_audit.txt
PERSPECTIVE-CONDITIONED SOFT-LABEL FEASIBILITY AUDIT

1. DATASET SUMMARY
------------------------------------------------------------------------------------------
Annotation rows: 49433
Unique comments: 19761
Unique annotators: 7851

2. ANNOTATION-LEVEL DEMOGRAPHIC COUNTS
------------------------------------------------------------------------------------------
Gender counts:
     gender_group  annotation_rows
            women            27716
              men            21112
       non_binary              362
prefer_not_to_say              189
    self_describe               54

Ideology counts:
        ideology_group  annotation_rows
               liberal            12437
               neutral             8222
      slightly_liberal             7844
     extremely_liberal             6667
          conservative             5772
 slightly_conservative             5418
extremely_conservative         